In [0]:
from pyspark.sql.functions import *

# 1. Read from Bronze table
bronze_df = spark.table("workspace.ibm.capstone_bronze_sales1")

# 2. Trim string columns & perform type casting
cleaned_df = (
    bronze_df
    # Trim strings
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("order_status", trim(col("order_status")))
    
    # Data type conversion
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("discount_pct", col("discount_pct").cast("double"))
    .withColumn("gross_amount", col("gross_amount").cast("double"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    .withColumn("net_amount", col("net_amount").cast("double"))
    
    # Date conversion
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
)

# 3. Apply Data Quality Checks & Flags (Stage 3)
dq_df = cleaned_df.withColumn(
    "quality_flag",
    when(col("customer_id").isNull() | (col("customer_id") == ""), "INVALID_CUSTOMER")
    .when(col("quantity").isNull() | (col("quantity") <= 0), "INVALID_QUANTITY")
    .when(col("net_amount").isNull() | (col("net_amount") < 0), "INVALID_AMOUNT")
    .when(col("product_id").isNull() | (col("product_id") == "UNKNOWN"), "INVALID_PRODUCT")
    .otherwise("VALID")
)

# 4. Filter out invalid records and add date transformation features
silver_df = (
    dq_df
    .filter(col("quality_flag") == "VALID") # Remove invalid business records
    .withColumn("year", year(col("order_date")))
    .withColumn("month", month(col("order_date")))
    .withColumn("month_name", date_format(col("order_date"), "MMMM"))
)


# 5. Write to Silver Delta Table (with overwriteSchema to prevent metadata mismatch errors)
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ibm.capstone_silver_sales1")

# 6. Store the invalid records
silver_rejected_df = (
    dq_df
    .filter(col("quality_flag") != "VALID") # Remove valid business records
)
silver_rejected_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ibm.capstone_silver_rejected_records")
# Preview valid records
print("Valid records written to Silver table.",silver_df.count())
print("Invalid records written to Silver rejected table.",silver_rejected_df.count())